In [ ]:
%pip install pandas seaborn matplotlib

In [ ]:
import pandas as pd
import os, re, sys
import numpy as np

## List-to-method correspondences

In [ ]:
def get_method_dict(file):
    method_dict = {'average' : 'average'}
    with open(file, 'r') as f:
        lines = [line.rstrip() for line in f.readlines()]
        i = 1
        for l in lines:
            match = re.search(r'^.+attrib=([^_]+)_mpd.+_thresh=([^_]+)_.+$', l)
            method = match.group(1)
            threshold = match.group(2)
            key = 'list' + str(i)
            method_dict[key] = method + '_' + threshold
            i += 1
    return method_dict


In [ ]:
bcms_setimes_methods = get_method_dict('randomized_lists/randomized_lists.bcms_setimes.txt')
bcms_twitter_methods = get_method_dict('randomized_lists/randomized_lists.bcms_twitter.txt')
estonian_voro_methods = get_method_dict('randomized_lists/randomized_lists.estonian_voro.txt')
finnish_suomi24_methods = get_method_dict('randomized_lists/randomized_lists.finnish.txt')
german_jodel_methods = get_method_dict('randomized_lists/randomized_lists.jodel.txt')
greek_dialects_methods = get_method_dict('randomized_lists/randomized_lists.greek.txt')
scandinavian_slide_methods = get_method_dict('randomized_lists/randomized_lists.scandinavian.txt')

In [ ]:
bcms_setimes_methods_df = pd.DataFrame.from_dict(bcms_setimes_methods, orient='index', columns=['method'])
bcms_setimes_methods_df

In [ ]:
bcms_twitter_methods_df = pd.DataFrame.from_dict(bcms_twitter_methods, orient='index', columns=['method'])
bcms_twitter_methods_df

In [ ]:
estonian_voro_methods_df = pd.DataFrame.from_dict(estonian_voro_methods, orient='index', columns=['method'])
estonian_voro_methods_df

In [ ]:
finnish_suomi24_methods_df = pd.DataFrame.from_dict(finnish_suomi24_methods, orient='index', columns=['method'])
finnish_suomi24_methods_df

In [ ]:
greek_dialects_methods_df = pd.DataFrame.from_dict(greek_dialects_methods, orient='index', columns=['method'])
greek_dialects_methods_df

In [ ]:
german_jodel_methods_df = pd.DataFrame.from_dict(german_jodel_methods, orient='index', columns=['method'])
german_jodel_methods_df

In [ ]:
scandinavian_slide_methods_df = pd.DataFrame.from_dict(scandinavian_slide_methods, orient='index', columns=['method'])
scandinavian_slide_methods_df

## Importing all lists and labelling them with list ID

In [ ]:
def make_dataset_df(folder):
    dfs = list()
    files = os.listdir(folder)
    for file in files:
        number = re.search(r'\d', file)
        if number:
            id = number.group()
        else:
            print("Skip file:", file)
            continue
        if file.endswith('.csv'):
            df = pd.read_csv(folder + '/' + file)
            df['file'] = "list" + id
            dfs.append(df)
        elif file.endswith('.txt'):
            df = pd.read_csv(folder + '/' + file, sep='\t')
            df['file'] = "list" + id
            dfs.append(df)
    full_df = pd.concat(dfs, ignore_index=True)
    return full_df

In [ ]:
def get_number_annots(df):
    number_annots = df.value_counts('file').to_frame()
    number_annots = number_annots.rename(columns={number_annots.columns[0]: "total"})
    return number_annots

In [ ]:
bcms_twitter_df = make_dataset_df('bcms_twitter')
bcms_twitter_df

In [ ]:
bcms_setimes_df = make_dataset_df('bcms_setimes')
bcms_setimes_df

In [ ]:
estonian_voro_df = make_dataset_df('eesti-voro')
estonian_voro_df

In [ ]:
german_jodel_df = make_dataset_df('german_jodel')
german_jodel_df

In [ ]:
german_jodel_df['other_variety'].value_counts()

In [ ]:
greek_dialects_df = make_dataset_df('greek_dialects')
greek_dialects_df

In [ ]:
finnish_suomi24_df = make_dataset_df('finnish')
finnish_suomi24_df

In [ ]:
scandinavian_slide_df = make_dataset_df('scandinavian')
scandinavian_slide_df

# Data preparation

Aim: data frame containing the percentage of lexical shibboleths, other shibboleths, non shibboleths and if applicable other annotations

## Preparing bcms_setimes dataframe

In [ ]:
number_annots = get_number_annots(bcms_setimes_df)
number_annots

In [ ]:
bcms_setimes_shib_distro = bcms_setimes_df[(bcms_setimes_df['shibboleth'] == 'yes') & (bcms_setimes_df['feature'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
bcms_setimes_shib_distro['nonlex shib'] = bcms_setimes_df[(bcms_setimes_df['shibboleth'] == 'yes') & (bcms_setimes_df['feature'] != 'lex')].groupby('file').size()
bcms_setimes_shib_distro['non shib'] = bcms_setimes_df[bcms_setimes_df['shibboleth'] == 'no'].groupby('file').size()
bcms_setimes_shib_distro['rest'] = bcms_setimes_df[(bcms_setimes_df['shibboleth'] != 'no') & (bcms_setimes_df['shibboleth'] != 'yes')].groupby('file').size()
bcms_setimes_shib_distro = pd.concat([bcms_setimes_shib_distro, number_annots], axis=1)
bcms_setimes_shib_distro['shibboleths: lexical'] = round(bcms_setimes_shib_distro['lex shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['shibboleths: other'] = round(bcms_setimes_shib_distro['nonlex shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['non shibboleths'] = round(bcms_setimes_shib_distro['non shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['other'] = round(bcms_setimes_shib_distro['rest'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['perc sum'] = bcms_setimes_shib_distro['shibboleths: lexical'] + bcms_setimes_shib_distro['shibboleths: other'] \
+ bcms_setimes_shib_distro['non shibboleths'] + bcms_setimes_shib_distro['other']
bcms_setimes_shib_distro.loc['average'] = round(bcms_setimes_shib_distro.mean(axis=0), 1)
bcms_setimes_shib_distro

In [ ]:
bcms_setimes_shib_distro = pd.concat([bcms_setimes_shib_distro, bcms_setimes_methods_df], axis=1)


In [ ]:
bcms_setimes_shib_distro

In [ ]:
bcms_setimes_shib_distro = bcms_setimes_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
bcms_setimes_shib_distro

In [ ]:
bcms_setimes_overview = bcms_setimes_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
bcms_setimes_overview

## Preparing bcms_twitter dataset

In [ ]:
number_annots = get_number_annots(bcms_twitter_df)
number_annots

In [ ]:
bcms_twitter_shib_distro = bcms_twitter_df[(bcms_twitter_df['shibboleth'] == 'yes') & (bcms_twitter_df['feature'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
bcms_twitter_shib_distro['nonlex shib'] = bcms_twitter_df[(bcms_twitter_df['shibboleth'] == 'yes') & (bcms_twitter_df['feature'] != 'lex')].groupby('file').size()
bcms_twitter_shib_distro['non shib'] = bcms_twitter_df[bcms_twitter_df['shibboleth'] == 'no'].groupby('file').size()
bcms_twitter_shib_distro['rest'] = bcms_twitter_df[(bcms_twitter_df['shibboleth'] != 'no') & (bcms_twitter_df['shibboleth'] != 'yes')].groupby('file').size()
bcms_twitter_shib_distro = pd.concat([bcms_twitter_shib_distro, number_annots], axis=1)
bcms_twitter_shib_distro['shibboleths: lexical'] = round(bcms_twitter_shib_distro['lex shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['shibboleths: other'] = round(bcms_twitter_shib_distro['nonlex shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['non shibboleths'] = round(bcms_twitter_shib_distro['non shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['other'] = round(bcms_twitter_shib_distro['rest'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['perc sum'] = bcms_twitter_shib_distro['shibboleths: lexical'] + bcms_twitter_shib_distro['shibboleths: other'] \
+ bcms_twitter_shib_distro['non shibboleths'] + bcms_twitter_shib_distro['other']
bcms_twitter_shib_distro.loc['average'] = round(bcms_twitter_shib_distro.mean(axis=0), 1)
bcms_twitter_shib_distro

In [ ]:
bcms_twitter_shib_distro = pd.concat([bcms_twitter_shib_distro, bcms_twitter_methods_df], axis = 1)
bcms_twitter_shib_distro

In [ ]:
bcms_twitter_shib_distro  = bcms_twitter_shib_distro.sort_values(by='method').reset_index(names=['list']).set_index('method')
bcms_twitter_shib_distro

In [ ]:
bcms_twitter_overview = bcms_twitter_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
bcms_twitter_overview

## Preparing german_jodel dataframe

In [ ]:
number_annots = get_number_annots(german_jodel_df)
number_annots

In [ ]:
german_jodel_df.value_counts('code')

In [ ]:
# Named entities are not shibboleths
# Only single-variety shibboleths count
german_jodel_df['shibboleth'] = 'no'
german_jodel_df.loc[(german_jodel_df['code'] != 'NE') & (german_jodel_df['code'] != 'unmarked') \
& (german_jodel_df['other_variety'].isna()) & (german_jodel_df['note'].isna()) , 'shibboleth'] = 'yes'
# sanity check:
german_jodel_df[(german_jodel_df['code'] == 'lex') & (german_jodel_df['shibboleth'] == 'no')]

In [ ]:
german_jodel_shib_distro = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
german_jodel_shib_distro['nonlex shib'] = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] != 'lex')].groupby('file').size()
german_jodel_shib_distro['non shib'] = german_jodel_df[german_jodel_df['shibboleth'] == 'no'].groupby('file').size()
german_jodel_shib_distro.loc[:,'sum'] = german_jodel_shib_distro.sum(axis=1)
german_jodel_shib_distro = pd.concat([german_jodel_shib_distro, number_annots], axis=1)
german_jodel_shib_distro['shibboleths: lexical'] = round(german_jodel_shib_distro['lex shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['shibboleths: other'] = round(german_jodel_shib_distro['nonlex shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['non shibboleths'] = round(german_jodel_shib_distro['non shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['perc_sum'] = german_jodel_shib_distro['shibboleths: lexical'] + german_jodel_shib_distro['shibboleths: other'] + german_jodel_shib_distro['non shibboleths']
german_jodel_shib_distro.loc['average'] = round(german_jodel_shib_distro.mean(axis=0), 1)

german_jodel_shib_distro

In [ ]:
german_jodel_shib_distro = pd.concat([german_jodel_shib_distro, german_jodel_methods_df], axis=1)
german_jodel_shib_distro

In [ ]:
german_jodel_shib_distro = german_jodel_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
german_jodel_shib_distro

In [ ]:
german_jodel_overview = german_jodel_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
german_jodel_overview

## Preparing greek_dialects dataframe

In [ ]:
greek_dialects_df[greek_dialects_df['file'] ==  'list2']

In [ ]:
greek_dialects_df.value_counts('feature')

In [ ]:
number_annots = get_number_annots(greek_dialects_df)
number_annots

In [ ]:
greek_dialects_shib_distro = greek_dialects_df[greek_dialects_df['feature'] == 'Lex'].groupby('file').size().to_frame(name='lex shib')
greek_dialects_shib_distro['nonlex shib'] = greek_dialects_df[(greek_dialects_df['feature'] != 'NE') & (greek_dialects_df['feature'] != 'Unmarked') & (greek_dialects_df['feature'] != 'Lex')].groupby('file').size()
greek_dialects_shib_distro['non shib'] = greek_dialects_df[(greek_dialects_df['feature'] == 'NE') | (greek_dialects_df['feature'] == 'Unmarked')].groupby('file').size()
greek_dialects_shib_distro.loc[:, 'sum'] = greek_dialects_shib_distro.sum(axis=1)
greek_dialects_shib_distro = pd.concat([greek_dialects_shib_distro, number_annots], axis=1)
greek_dialects_shib_distro['shibboleths: lexical'] = round(greek_dialects_shib_distro['lex shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['shibboleths: other'] = round(greek_dialects_shib_distro['nonlex shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['non shibboleths'] = round(greek_dialects_shib_distro['non shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['perc_sum'] = greek_dialects_shib_distro['shibboleths: lexical'] + greek_dialects_shib_distro['shibboleths: other'] \
+ greek_dialects_shib_distro['non shibboleths']
greek_dialects_shib_distro.loc['average'] = round(greek_dialects_shib_distro.mean(axis=0) , 1)

greek_dialects_shib_distro

In [ ]:
greek_dialects_shib_distro = pd.concat([greek_dialects_shib_distro, greek_dialects_methods_df], axis=1)
greek_dialects_shib_distro

In [ ]:
greek_dialects_shib_distro = greek_dialects_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
greek_dialects_shib_distro

In [ ]:
greek_dialects_overview = greek_dialects_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
greek_dialects_overview

## Preparing finnish_suomi24 dataframe

In [ ]:
finnish_suomi24_df

In [ ]:
finnish_suomi24_df.value_counts('label')

In [ ]:
number_annots = get_number_annots(finnish_suomi24_df)
number_annots

In [ ]:
finnish_suomi24_shib_distro = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'lex'].groupby('file').size().to_frame(name='lex shib')
finnish_suomi24_shib_distro['nonlex shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'other'].groupby('file').size()
finnish_suomi24_shib_distro['non shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] ==  'not'].groupby('file').size()
finnish_suomi24_shib_distro.loc[:, 'sum'] = finnish_suomi24_shib_distro.sum(axis=1)
finnish_suomi24_shib_distro = pd.concat([finnish_suomi24_shib_distro, number_annots], axis=1)
finnish_suomi24_shib_distro['shibboleths: lexical'] = round(finnish_suomi24_shib_distro['lex shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['shibboleths: other'] = round(finnish_suomi24_shib_distro['nonlex shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['non shibboleths'] = round(finnish_suomi24_shib_distro['non shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['perc_sum'] = finnish_suomi24_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']].sum(axis=1)
finnish_suomi24_shib_distro.loc['average'] = round(finnish_suomi24_shib_distro.mean(axis=0), 1)


finnish_suomi24_shib_distro

In [ ]:
finnish_suomi24_shib_distro = pd.concat([finnish_suomi24_shib_distro, finnish_suomi24_methods_df], axis=1)
finnish_suomi24_shib_distro

In [ ]:
finnish_suomi24_shib_distro = finnish_suomi24_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
finnish_suomi24_shib_distro

In [ ]:
finnish_suomi24_overview = finnish_suomi24_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
finnish_suomi24_overview

## Prepare estonian-voro dataframe

In [ ]:
estonian_voro_df

In [ ]:
number_annots = get_number_annots(estonian_voro_df)
number_annots

In [ ]:
estonian_voro_df[estonian_voro_df['Lexicon'].str.contains(r'x|\?', na=False)]

In [ ]:
estonian_voro_df["unk"] = 0
estonian_voro_df.loc[(estonian_voro_df['Unknown'].str.contains('x', na=False) | estonian_voro_df['Other'].str.contains('x', na=False)), "unk"] = 1
estonian_voro_df["noshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & estonian_voro_df['Unmarked'].str.contains('x', na=False)), "noshib"] = 1
estonian_voro_df["lexshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & (estonian_voro_df["noshib"] == 0) & estonian_voro_df['Lexicon'].str.contains('x', na=False)), "lexshib"] = 1
estonian_voro_df["nonlexshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & (estonian_voro_df["noshib"] == 0) & (estonian_voro_df["lexshib"] == 0) & estonian_voro_df['Phon-Morph-infl-Morph-der-Spelling'].str.contains(r'x|s', na=False)), "nonlexshib"] = 1

estonian_voro_df.loc[estonian_voro_df["unk"] + estonian_voro_df["noshib"] + estonian_voro_df["lexshib"] + estonian_voro_df["nonlexshib"] == 0, "unk"] += 1

estonian_voro_df[estonian_voro_df["unk"] >= 1]

In [ ]:
estonian_voro_shib_distro = estonian_voro_df[estonian_voro_df['lexshib']==1].groupby('file').size().to_frame(name="lex shib")
estonian_voro_shib_distro['non lex shib'] = estonian_voro_df[estonian_voro_df['nonlexshib']==1].groupby('file').size()
estonian_voro_shib_distro['non shib'] = estonian_voro_df[estonian_voro_df['noshib']==1].groupby('file').size()
estonian_voro_shib_distro['rest'] = estonian_voro_df[estonian_voro_df['unk']==1].groupby('file').size()
estonian_voro_shib_distro.loc[:, 'sum'] = estonian_voro_shib_distro.sum(axis=1)

estonian_voro_shib_distro = pd.concat([estonian_voro_shib_distro, number_annots], axis=1)
estonian_voro_shib_distro['shibboleths: lexical'] = round(estonian_voro_shib_distro['lex shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['shibboleths: other'] = round(estonian_voro_shib_distro['non lex shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['non shibboleths'] = round(estonian_voro_shib_distro['non shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['other'] = round(estonian_voro_shib_distro['rest'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)

estonian_voro_shib_distro.loc[:, 'perc_sum'] = estonian_voro_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']].sum(axis=1)
estonian_voro_shib_distro.loc['average'] = round(estonian_voro_shib_distro.mean(axis=0), 1)

estonian_voro_shib_distro

In [ ]:
estonian_voro_shib_distro = pd.concat([estonian_voro_shib_distro, estonian_voro_methods_df], axis=1)
estonian_voro_shib_distro

In [ ]:
estonian_voro_shib_distro = estonian_voro_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
estonian_voro_shib_distro

In [ ]:
estonian_voro_overview = estonian_voro_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
estonian_voro_overview

## Prepare scandinavian_slide dataframe

In [ ]:
scandinavian_slide_df

In [ ]:
scandinavian_slide_df[(scandinavian_slide_df['NE'] == True) & (scandinavian_slide_df['NonLexShib'] == True)]

In [ ]:
number_annots = get_number_annots(scandinavian_slide_df)
number_annots

In [ ]:
scandinavian_slide_shib_distro = scandinavian_slide_df[scandinavian_slide_df['LexShib'] == True].groupby('file').size().to_frame(name='lex shib')
scandinavian_slide_shib_distro['nonlex shib'] = scandinavian_slide_df[scandinavian_slide_df['NonLexShib'] == True].groupby('file').size()
scandinavian_slide_shib_distro['non shib'] = scandinavian_slide_df[(scandinavian_slide_df['NotShib'] ==  True)].groupby('file').size()
scandinavian_slide_shib_distro['rest'] = scandinavian_slide_df[(scandinavian_slide_df['Typo_Unk'] == True)].groupby('file').size()
scandinavian_slide_shib_distro.loc[:, 'sum'] = scandinavian_slide_shib_distro.sum(axis=1)
scandinavian_slide_shib_distro = pd.concat([scandinavian_slide_shib_distro, number_annots], axis=1)

scandinavian_slide_shib_distro['shibboleths: lexical'] = round(scandinavian_slide_shib_distro['lex shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['shibboleths: other'] = round(scandinavian_slide_shib_distro['nonlex shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['non shibboleths'] = round(scandinavian_slide_shib_distro['non shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['other'] = round(scandinavian_slide_shib_distro['rest'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)

scandinavian_slide_shib_distro.loc[:, 'perc_sum'] = scandinavian_slide_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']].sum(axis=1)

scandinavian_slide_shib_distro.loc['average'] = round(scandinavian_slide_shib_distro.mean(axis=0), 1)

scandinavian_slide_shib_distro

In [ ]:
scandinavian_slide_shib_distro = pd.concat([scandinavian_slide_shib_distro, scandinavian_slide_methods_df], axis=1)
scandinavian_slide_shib_distro

In [ ]:
scandinavian_slide_shib_distro = scandinavian_slide_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
scandinavian_slide_shib_distro

In [ ]:
scandinavian_slide_overview = scandinavian_slide_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
scandinavian_slide_overview

## Generate graphs

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
german_jodel_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : german_jodel").figure.savefig('german_jodel_shib_overview.pdf', bbox_inches='tight')

In [ ]:
bcms_twitter_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : bcms_twitter").figure.savefig('bcms_twitter_shib_overview.pdf', bbox_inches='tight')

In [ ]:
bcms_setimes_overview.plot(kind='bar', colormap="summer", stacked=True, title="Shibboleth distribution overview : bcms_setimes").figure.savefig('bcms_setimes_shib_overview.pdf', bbox_inches='tight')

In [ ]:
greek_dialects_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : greek_dialects").figure.savefig('greek_dialects_shib_overview.pdf', bbox_inches='tight')

In [ ]:
finnish_suomi24_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : finnish_suomi24").figure.savefig('finnish_suomi24_shib_overview.pdf', bbox_inches='tight')

In [ ]:
scandinavian_slide_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : scandinavian_slide").figure.savefig('scandinavian_slide_shib_overview.pdf', bbox_inches='tight')

In [ ]:
estonian_voro_overview.plot(kind='bar', stacked=True, colormap="summer", title="Shibboleth distribution overview : estonian_voro").figure.savefig('estonian_voro_shib_overview.pdf', bbox_inches='tight')

## Generating csv tables with main results

In [ ]:
bcms_setimes_overview.to_csv('bcms_setimes_overview.csv')
bcms_twitter_overview.to_csv('bcms_twitter_overview.csv')
finnish_suomi24_overview.to_csv('finnish_suomi24_overview.csv')
german_jodel_overview.to_csv('german_jodel_overview.csv')
greek_dialects_overview.to_csv('greek_dialects_overview.csv')
scandinavian_slide_overview.to_csv('scandinavian_slide_overview.csv')
estonian_voro_overview.to_csv('estonian_voro_overview.csv')

# Analysis by method 

In [ ]:
bcms_setimes_overview.loc[:, 'dataset'] = 'bcms_setimes'
bcms_setimes_overview

In [ ]:
bcms_twitter_overview.loc[:, 'dataset'] = 'bcms_twitter'
bcms_twitter_overview

In [ ]:
finnish_suomi24_overview.loc[:, 'dataset'] = 'finnish_suomi24'
finnish_suomi24_overview

In [ ]:
german_jodel_overview.loc[:, 'dataset'] = 'german_jodel'
german_jodel_overview

In [ ]:
greek_dialects_overview.loc[:, 'dataset'] = 'greek_dialects'
greek_dialects_overview

In [ ]:
scandinavian_slide_overview.loc[:, 'dataset'] = 'scandinavian_slide'
scandinavian_slide_overview

In [ ]:
estonian_voro_overview.loc[:, 'dataset'] = 'estonian_voro'
estonian_voro_overview

In [ ]:
all_overview = pd.concat([bcms_twitter_overview, bcms_setimes_overview, finnish_suomi24_overview, \
                         german_jodel_overview, greek_dialects_overview, scandinavian_slide_overview, estonian_voro_overview], axis=0).reset_index(names=['method'])
#all_overview['method'] = all_overview['method'].str.split("_", expand=True)[0].str.upper() + ", p=" + all_overview['method'].str.split("_", expand=True)[1]
all_overview['method'] = all_overview['method'].map({
	"ig_0.1": "IG, p=0.1",
	"ig_0.6": "IG, p=0.6",
	"lime_0.1": "LIME, p=0.1",
	"lime_0.6": "LIME, p=0.6",
	"loo_0.1": "LOO, p=0.1",
	"loo_0.6": "LOO, p=0.6",
	"shap_0.1": "SHAP, p=0.1",
	"shap_0.6": "SHAP, p=0.6",
	"average": "average"
})
all_overview["other"] = all_overview["other"].fillna(0)
all_overview

In [ ]:
all_overview[all_overview['method'] == 'IG, p=0.1'].set_index('dataset')

## Generate graphs by method

In [ ]:
import matplotlib.cm as cm
import numpy as np

In [ ]:
mean_lex_df = all_overview[all_overview['method'] == 'average'] #reset_index(names=['dataset'])
mean_lex_df.set_index('dataset', inplace=True)
mean_lex_df

In [ ]:
mean_ev = mean_lex_df.loc['estonian_voro', 'shibboleths: lexical']
mean_twit = mean_lex_df.loc['bcms_setimes', 'shibboleths: lexical']
mean_set = mean_lex_df.loc['bcms_twitter', 'shibboleths: lexical']
mean_suomi24 = mean_lex_df.loc['finnish_suomi24', 'shibboleths: lexical']
mean_jodel = mean_lex_df.loc['german_jodel', 'shibboleths: lexical']
mean_greek = mean_lex_df.loc['greek_dialects', 'shibboleths: lexical']
mean_slide = mean_lex_df.loc['scandinavian_slide', 'shibboleths: lexical']

In [ ]:
all_overview = all_overview[all_overview['method'] != 'average']
all_overview


In [ ]:
sns.set_style("whitegrid")

fig, axes = plt.subplots(nrows=1, ncols=3, sharey='row', figsize=(7,4))
all_overview[all_overview['dataset'] == 'bcms_setimes'].set_index('method').plot(kind='bar', stacked=True, title="Croatian-Serbian",  colormap="summer", ax=axes[1], legend=False)
all_overview[all_overview['dataset'] == 'estonian_voro'].set_index('method').plot(kind='bar', stacked=True, title="Estonian-Võro",  colormap="summer", ax=axes[2], legend=False)
all_overview[all_overview['dataset'] == 'scandinavian_slide'].set_index('method').plot(kind='bar', stacked=True, title="Scandinavian",  colormap="summer", ax=axes[0], legend=False)

axes[0].set_yticks(range(0, 105, 10))
axes[0].set_ylim(0, 105)

axes[0].set_xlabel(None)
axes[1].set_xlabel(None)
axes[2].set_xlabel(None)

## adding mean line
axes[0].axhline(mean_slide, color='darkorange', linestyle='-')
#axes[0].text(0, mean_slide + 0.2, f'{mean_slide:.2f}', color='darkorange', weight='bold')
axes[1].axhline(mean_set, color='darkorange', linestyle='-', label='Mean % of lexical shibboleths')
#axes[1].text(0, mean_set + 0.2, f'{mean_set:.2f}', color='darkorange')
axes[2].axhline(mean_ev, color='darkorange', linestyle='-')
#axes[2].text(0, mean_ev + 0.2, f'{mean_ev:.2f}', color='darkorange')

ax=axes[1]
handles, labels = ax.get_legend_handles_labels()
print(labels)
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.3, -0.4), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.3, hspace=0.5
                   )
fig.figure.savefig('experiment1-overview.pdf', bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=4, sharey='row', figsize=(10,4))
all_overview[all_overview['dataset'] == 'bcms_twitter'].set_index('method').plot(kind='bar', stacked=True, title="bcms_twitter",  colormap="summer", ax=axes[0], legend=False)
all_overview[all_overview['dataset'] == 'finnish_suomi24'].set_index('method').plot(kind='bar', stacked=True, title="finnish_suomi24",  colormap="summer", ax=axes[1], legend=False)
all_overview[all_overview['dataset'] == 'german_jodel'].set_index('method').plot(kind='bar', stacked=True, title="german_jodel",  colormap="summer", ax=axes[2], legend=False)
all_overview[all_overview['dataset'] == 'greek_dialects'].set_index('method').plot(kind='bar', stacked=True, title="greek_dialects",  colormap="summer", ax=axes[3], legend=False)                                                                                 
axes[0].set_yticks(range(0, 105, 10))
axes[0].set_ylim(0, 105)

axes[0].set_xlabel(None)
axes[1].set_xlabel(None)
axes[2].set_xlabel(None)
axes[3].set_xlabel(None)

axes[0].axhline(mean_twit, color='darkorange', linestyle='-')
axes[1].axhline(mean_suomi24, color='darkorange', linestyle='-')
axes[2].axhline(mean_jodel, color='darkorange', linestyle='-', label='Mean % of lexical shibboleths')
axes[3].axhline(mean_greek, color='darkorange', linestyle='-')

ax=axes[2]
handles, labels = ax.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.2, -0.4), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.2, hspace=0.5
                   )
fig.figure.savefig('experiment2-overview-per_dataset.pdf', bbox_inches='tight')

### RESULTS PER DATASET, PER CLASS

## BCMS SETIMES

In [ ]:
bcms_setimes_df2 = bcms_setimes_df.set_index('file', drop=True)
bcms_setimes_df2

In [ ]:
bcms_setimes_methods_df

In [ ]:
bcms_setimes_df2 = bcms_setimes_df2.join(bcms_setimes_methods_df, how='left', on=None)

In [ ]:
bcms_setimes_df2

In [ ]:
methods = ['ig_0.1', 'ig_0.6', 'lime_0.1', 'lime_0.6', 'loo_0.1', 'loo_0.6',
       'shap_0.1', 'shap_0.6']
classes = ['hr', 'sr']

counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = bcms_setimes_df2[(bcms_setimes_df2['method'] == m) & (bcms_setimes_df2['label'] == c)]
        lex = len(work_df[(work_df['shibboleth'] == 'yes') & (work_df['feature'] == 'lex')])
        nonlex = len(work_df[(work_df['shibboleth'] == 'yes') & (work_df['feature'] != 'lex')])
        nonshib = len(work_df[(work_df['shibboleth'] == 'no')])
        other = len(work_df[(work_df['shibboleth'] != 'yes') & (work_df['shibboleth'] != 'no')])
        sum = lex + nonlex + nonshib + other
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum])


In [ ]:
counts_df = pd.DataFrame.from_dict(counts, orient='index').stack().to_frame()
counts_df[['Lexical', 'Non-lexical', 'Non shibboleth', 'Other', 'sum']] = counts_df.loc[:, 0].to_list()
counts_df = counts_df.drop(columns=[counts_df.columns[0]]).drop(columns=['sum'])
counts_df

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, sharey='row', figsize=(10,8))

idx = -1
for i in range(0,2):
    for j in range(0,4):
        idx += 1  
        m = methods[idx]
        counts_df.loc[m].plot(kind='bar', stacked=True, title=m, colormap="summer", ax=axes[i,j], legend=False)
        mean = counts_df.loc[m]['Lexical'].mean()
       # if i == 1 and j == 3:
        axes[i,j].axhline(mean, color='darkorange', linestyle='-', label='Mean % of lexical shibboleths')

axes[0, 0].set_yticks(range(0, 105, 10))
axes[0, 0].set_ylim(0, 105)

axes[1, 0].set_yticks(range(0, 105, 10))
axes[1, 0].set_ylim(0, 105)

ax=axes[1,3]
handles, labels = ax.get_legend_handles_labels()
#fig.legend(handles, labels, loc='upper center')
plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.2, -0.1), bbox_transform=plt.gcf().transFigure)
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.2, hspace=0.3
                   )

fig.figure.savefig('bcms_setimes_perclass.pdf', bbox_inches='tight')

In [ ]:
def make_graphs_per_class(dict, ds):
    
    counts = dict
    outfile = ds + '_perclass.pdf'

    counts_df = pd.DataFrame.from_dict(counts, orient='index').stack().to_frame()
    counts_df[['Lexical', 'Non-lexical', 'Non shibboleth', 'Other', 'sum']] = counts_df.loc[:, 0].to_list()
    counts_df = counts_df.drop(columns=[counts_df.columns[0]]).drop(columns=['sum'])
    counts_df
    
    fig, axes = plt.subplots(nrows=2, ncols=4, sharey='row', figsize=(10,8))
    
    idx = -1
    for i in range(0,2):
        for j in range(0,4):
            idx += 1  
            m = methods[idx]
            counts_df.loc[m].plot(kind='bar', stacked=True, title=m, colormap="summer", ax=axes[i,j], legend=False)
            mean = counts_df.loc[m]['Lexical'].mean()
           # if i == 1 and j == 3:
            axes[i,j].axhline(mean, color='darkorange', linestyle='-', label='Mean % of lexical shibboleths')
    
    axes[0, 0].set_yticks(range(0, 105, 10))
    axes[0, 0].set_ylim(0, 105)
    
    axes[1, 0].set_yticks(range(0, 105, 10))
    axes[1, 0].set_ylim(0, 105)
    
    ax=axes[1,3]
    handles, labels = ax.get_legend_handles_labels()
    #fig.legend(handles, labels, loc='upper center')
    plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.2, -0.1), bbox_transform=plt.gcf().transFigure)
    plt.subplots_adjust(left=0.1, right=0.9, 
                        top=0.9, bottom=0.1, 
                        wspace=0.2, hspace=0.3
                       )
    
    fig.figure.savefig(outfile, bbox_inches='tight')

## BCMS TWITTER

In [ ]:
bcms_twitter_df2 = bcms_twitter_df.set_index('file', drop=True)
bcms_twitter_df2 = bcms_twitter_df2.join(bcms_twitter_methods_df, how='left' or 'inner', on=None)

In [ ]:
bcms_twitter_df2

In [ ]:
classes = bcms_twitter_df2['label'].sort_values().unique()
classes

In [ ]:
methods = bcms_twitter_df2['method'].sort_values().unique()

In [ ]:
methods

In [ ]:
counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = bcms_twitter_df2[(bcms_twitter_df2['method'] == m) & (bcms_twitter_df2['label'] == c)]
        lex = len(work_df[(work_df['shibboleth'] == 'yes') & (work_df['feature'] == 'lex')])
        nonlex = len(work_df[(work_df['shibboleth'] == 'yes') & (work_df['feature'] != 'lex')])
        nonshib = len(work_df[(work_df['shibboleth'] == 'no')])
        other = len(work_df[(work_df['shibboleth'] != 'yes') & (work_df['shibboleth'] != 'no')])
        sum = lex + nonlex + nonshib + other
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum])
counts

In [ ]:
make_graphs_per_class(counts, 'bcms_twitter')

### SCANDINAVIAN_SLIDE

In [ ]:
scandinavian_slide_df2 = scandinavian_slide_df.set_index('file', drop=True)
scandinavian_slide_df2 = scandinavian_slide_df2.join(scandinavian_slide_methods_df, how='left', on=None)
scandinavian_slide_df2['method'].value_counts()

In [ ]:
classes = scandinavian_slide_df2['label'].sort_values().unique()
classes

In [ ]:
methods = scandinavian_slide_df2['method'].sort_values().unique()
methods

In [ ]:
counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = scandinavian_slide_df2[(scandinavian_slide_df2['method'] == m) & (scandinavian_slide_df2['label'] == c)]
        lex = len(work_df[work_df['LexShib'] == True])
        nonlex = len(work_df[work_df['NonLexShib'] == True])
        nonshib = len(work_df[work_df['NotShib'] == True])
        other = len(work_df[work_df['Typo_Unk'] == True])
        sum = lex + nonlex + nonshib + other
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum])
counts


In [ ]:
make_graphs_per_class(counts, 'scandinavian_slide')

### PREPARING ESTONIAN-VORO 

In [ ]:
estonian_voro_df2 = estonian_voro_df.set_index('file', drop=True)
estonian_voro_df2 = estonian_voro_df2.join(estonian_voro_methods_df, how='left', on=None)
estonian_voro_df2['method'].value_counts()

In [ ]:
classes = estonian_voro_df2['label'].sort_values().unique()
classes

In [ ]:
methods = estonian_voro_df2['method'].sort_values().unique()
methods

In [ ]:
counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = estonian_voro_df2[(estonian_voro_df2['method'] == m) & (estonian_voro_df2['label'] == c)]
        lex = len(work_df[work_df['lexshib'] == 1])
        nonlex = len(work_df[work_df['nonlexshib'] == 1])
        nonshib = len(work_df[work_df['noshib'] == 1])
        other = len(work_df[work_df['unk'] == 1])
        sum = lex + nonlex + nonshib + other
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum])
counts

#estonian_voro_shib_distro = estonian_voro_df[estonian_voro_df['lexshib']==1].groupby('file').size().to_frame(name="lex shib")
#estonian_voro_shib_distro['non lex shib'] = estonian_voro_df[estonian_voro_df['nonlexshib']==1].groupby('file').size()
#estonian_voro_shib_distro['non shib'] = estonian_voro_df[estonian_voro_df['noshib']==1].groupby('file').size()
#estonian_voro_shib_distro['rest'] = estonian_voro_df[estonian_voro_df['unk']==1].groupby('file').size()

In [ ]:
make_graphs_per_class(counts, 'estonian_voro')

### PREPARING JODEL DATA

In [ ]:
german_jodel_df2 = german_jodel_df.set_index('file', drop=True)
german_jodel_df2 = german_jodel_df2.join(german_jodel_methods_df, how='left', on=None)
german_jodel_df2['method'].value_counts()

In [ ]:
classes = german_jodel_df2['label'].sort_values().unique()
classes

In [ ]:
methods = german_jodel_df2['method'].sort_values().unique()
methods

In [ ]:
counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = german_jodel_df2[(german_jodel_df2['method'] == m) & (german_jodel_df2['label'] == c)]
        lex = len(work_df[(work_df['shibboleth'] == 'yes') & (work_df['code'] == 'lex')])
        nonlex = len(work_df[(work_df['shibboleth'] == 'yes') & (work_df['code'] != 'lex')])
        nonshib = len(work_df[work_df['shibboleth'] == 'no'])
        other = 0
        sum = lex + nonlex + nonshib + other
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum])
counts

#german_jodel_shib_distro = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
#german_jodel_shib_distro['nonlex shib'] = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] != 'lex')].groupby('file').size()
#german_jodel_shib_distro['non shib'] = german_jodel_df[german_jodel_df['shibboleth'] == 'no'].groupby('file').size()

In [ ]:
make_graphs_per_class(counts, 'german_jodel')

### PREPARING GREEK DATA

In [ ]:
greek_dialects_df2 = greek_dialects_df.set_index('file', drop=True)
greek_dialects_df2 = greek_dialects_df2.join(greek_dialects_methods_df, how='left', on=None)
greek_dialects_df2['method'].value_counts()

In [ ]:
classes = greek_dialects_df2['label'].sort_values().unique()
classes

In [ ]:
methods = greek_dialects_df2['method'].sort_values().unique()
methods

In [ ]:
counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = greek_dialects_df2[(greek_dialects_df2['method'] == m) & (greek_dialects_df2['label'] == c)]
        lex = len(work_df[work_df['feature'] == 'Lex'])
        nonlex = len(work_df[(work_df['feature'] != 'NE') & (work_df['feature'] != 'Unmarked') & (work_df['feature'] != 'Lex')])
        nonshib = len(work_df[(work_df['feature'] == 'NE') | (work_df['feature'] == 'Unmarked')])
        other = 0
        sum = lex + nonlex + nonshib + other
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum])
counts

#greek_dialects_shib_distro = greek_dialects_df[greek_dialects_df['feature'] == 'Lex'].groupby('file').size().to_frame(name='lex shib')
#greek_dialects_shib_distro['nonlex shib'] = greek_dialects_df[(greek_dialects_df['feature'] != 'NE') & (greek_dialects_df['feature'] != 'Unmarked') & (greek_dialects_df['feature'] != 'Lex')].groupby('file').size()
#greek_dialects_shib_distro['non shib'] = greek_dialects_df[(greek_dialects_df['feature'] == 'NE') | (greek_dialects_df['feature'] == 'Unmarked')].groupby('file').size()

In [ ]:
make_graphs_per_class(counts, 'greek_dialects')

### PREPARING FINNISH DATA

In [ ]:
finnish_suomi24_df2 = finnish_suomi24_df.set_index('file', drop=True)
finnish_suomi24_df2 = finnish_suomi24_df2.join(finnish_suomi24_methods_df, how='left', on=None)
finnish_suomi24_df2['method'].value_counts()

In [ ]:
sums = finnish_suomi24_df2.groupby(['method', 'label']).size()
sums

In [ ]:
methods = finnish_suomi24_df2['method'].sort_values().unique()
methods

In [ ]:
classes = finnish_suomi24_df2['label'].sort_values().unique()
classes

In [ ]:
counts = {}

for m in methods:
    counts[m] = {}
    for c in classes:
        counts[m][c] = []
        work_df = finnish_suomi24_df2[(finnish_suomi24_df2['method'] == m) & (finnish_suomi24_df2['label'] == c)]
        lex = len(work_df[work_df['lex_other_not'] == 'lex'])
        nonlex = len(work_df[work_df['lex_other_not'] == 'other'])
        nonshib = len(work_df[work_df['lex_other_not'] == 'not'])
        other = 0
       # sum = lex + nonlex + nonshib + other
        lex = round(lex / sums[m][c] * 100, 1)
        nonlex = round(nonlex / sums[m][c] * 100, 1)
        nonshib = round(nonshib / sums[m][c] * 100, 1)
        sum2 = lex + nonlex + nonshib
        
        counts[m][c].extend([lex, nonlex, nonshib, other, sum2])
counts

# finnish_suomi24_shib_distro = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'lex'].groupby('file').size().to_frame(name='lex shib')
# finnish_suomi24_shib_distro['nonlex shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'other'].groupby('file').size()
# finnish_suomi24_shib_distro['non shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] ==  'not'].groupby('file').size()

In [ ]:
make_graphs_per_class(counts, 'finnish_suomi24')

In [ ]:
def make_graphs_per_class(dict, ds):
    
    counts = dict
    outfile = ds + '_perclass.pdf'

    counts_df = pd.DataFrame.from_dict(counts, orient='index').stack().to_frame()
    counts_df[['Lexical', 'Non-lexical', 'Non shibboleth', 'Other', 'sum']] = counts_df.loc[:, 0].to_list()
    counts_df = counts_df.drop(columns=[counts_df.columns[0]]).drop(columns=['sum'])
    counts_df
    
    fig, axes = plt.subplots(nrows=2, ncols=4, sharey='row', figsize=(10,8))
    
    idx = -1
    for i in range(0,2):
        for j in range(0,4):
            idx += 1  
            m = methods[idx]
            counts_df.loc[m].plot(kind='bar', stacked=True, title=m, colormap="summer", ax=axes[i,j], legend=False)
            mean = counts_df.loc[m]['Lexical'].mean()
           # if i == 1 and j == 3:
            axes[i,j].axhline(mean, color='darkorange', linestyle='-', label='Mean % of lexical shibboleths')
    
    axes[0, 0].set_yticks(range(0, 105, 10))
    axes[0, 0].set_ylim(0, 105)
    
    axes[1, 0].set_yticks(range(0, 105, 10))
    axes[1, 0].set_ylim(0, 105)
    
    ax=axes[1,3]
    handles, labels = ax.get_legend_handles_labels()
    #fig.legend(handles, labels, loc='upper center')
    plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.2, -0.1), bbox_transform=plt.gcf().transFigure)
    plt.subplots_adjust(left=0.1, right=0.9, 
                        top=0.9, bottom=0.1, 
                        wspace=0.2, hspace=0.3
                       )
    
    fig.figure.savefig(outfile, bbox_inches='tight')

### LEVELS OF LINGUISTIC STRUCTURE

In [ ]:
def make_graphs_per_level(df, methods, classes, dataset):
    
    work_df = df
    methods = methods
    classes = classes
    ds = dataset
    dfs = []
    outfile = dataset + '_perlevel.pdf'

    fig, axes = plt.subplots(nrows=2, ncols=4, sharey='row', figsize=(10,8))
        
    
    i = 0
    j = -1
    for m in methods:
        j += 1
        for c in classes:
            try: ## added
                tmp = work_df.loc[m, c] ## indendted
                tmp['sum'] = tmp['count'].sum()
                tmp['perc'] = round(tmp['count'] / tmp['sum'] * 100, 1)
                tmp = tmp.drop(columns=['count', 'sum']).transpose()
                tmp['label'] = c
                tmp = tmp.set_index('label')
                dfs.append(tmp)
            except KeyError:
                pass
    
        df_to_plot = pd.concat(dfs).sort_index(axis=1)    
        df_to_plot.plot(kind='bar', stacked=True, title = m, colormap='summer', ax = axes[i,j], legend=False)
    
        dfs = []
        
        if j == 3:
            i += 1
            j = -1
    
    axes[0, 0].set_yticks(range(0, 105, 10))
    axes[0, 0].set_ylim(0, 105)
    
    axes[1, 0].set_yticks(range(0, 105, 10))
    axes[1, 0].set_ylim(0, 105)
    
    axes[0,0].set_xlabel(None)
    axes[0,1].set_xlabel(None)
    axes[0,2].set_xlabel(None)
    axes[0,3].set_xlabel(None)
    axes[1,0].set_xlabel(None)
    axes[1,1].set_xlabel(None)
    axes[1,2].set_xlabel(None)
    axes[1,3].set_xlabel(None)
    
    ax=axes[0,2]
    handles, labels = ax.get_legend_handles_labels()
    plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.97, 0.7), bbox_transform=plt.gcf().transFigure)
    plt.subplots_adjust(left=0.1, right=0.9, 
                        top=0.9, bottom=0.1, 
                        wspace=0.2, hspace=0.3
                       )
    
    fig.figure.savefig(outfile, bbox_inches='tight')

In [ ]:
def make_sel_graph_per_level(df, method, classes, dataset):
    
    work_df = df
    m = method
    classes = classes
    ds = dataset
    dfs = []
    outfile = dataset + '_perlevel_sel.pdf'

    #fig  = plt.plot(figsize=(5,4))
    
      
    for c in classes:
        try: ## added
            tmp = work_df.loc[m, c] ## indented
            tmp['sum'] = tmp['count'].sum()
            tmp['perc'] = round(tmp['count'] / tmp['sum'] * 100, 1)
            tmp = tmp.drop(columns=['count', 'sum']).transpose()
            tmp['label'] = c
            tmp = tmp.set_index('label')
            dfs.append(tmp)
        except KeyError:
            pass
    
    df_to_plot = pd.concat(dfs).sort_index(axis=1)    
    df_to_plot.plot(kind='bar', stacked=True, title = ds, colormap='summer', figsize=(5,4), xlabel=method).legend(bbox_to_anchor=(1.0, 1.0)).figure.savefig(outfile, bbox_inches='tight')
    
    
   # plt.set_yticks(range(0, 105, 10))
   # plt.set_ylim(0, 105)
    
#    axes[1, 0].set_yticks(range(0, 105, 10))
#    axes[1, 0].set_ylim(0, 105)
#    
#    axes[0,0].set_xlabel(None)
#    axes[0,1].set_xlabel(None)
#    axes[0,2].set_xlabel(None)
#    axes[0,3].set_xlabel(None)
#    axes[1,0].set_xlabel(None)
#    axes[1,1].set_xlabel(None)
#    axes[1,2].set_xlabel(None)
#    axes[1,3].set_xlabel(None)
    
#    ax=axes[0,2]
#    handles, labels = ax.get_legend_handles_labels()
#    plt.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.97, 0.7), bbox_transform=plt.gcf().transFigure)
#   # plt.subplots_adjust(left=0.1, right=0.9, 
#                        top=0.9, bottom=0.1, 
#                        wspace=0.2, hspace=0.3
#                       )
    
   # plt.figure.savefig(outfile, bbox_inches='tight')

### PREPARING BCMS SETIMES

In [ ]:
bcms_setimes_df2

In [ ]:
f1 = bcms_setimes_df2[['feature', 'label']]
len(f1)

In [ ]:
f2 = bcms_setimes_df2[['feature2', 'label']].dropna()
f2['feature'] = f2['feature2']
f2 = f2.drop(columns=['feature2'])
len(f2)

In [ ]:
feature_df = pd.concat([f1, f2])
feature_df['file'] = feature_df.index
feature_df = feature_df.reset_index().drop(columns=('index'))
feature_df = feature_df[feature_df.feature.str.contains('phon|lex|morph-d|morph-f|spel')]
feature_df = feature_df.join(bcms_setimes_methods_df, how='left', on='file').drop(columns=('file'))
feature_df
# per method, per class

In [ ]:
methods = feature_df['method'].sort_values().unique()
methods

In [ ]:
classes = feature_df['label'].sort_values().unique()
classes

In [ ]:
bcms_setimes_levels_df = feature_df.groupby(['method', 'label', 'feature']).size().to_frame(name='count')
bcms_setimes_levels_df

In [ ]:
make_graphs_per_level(bcms_setimes_levels_df, methods, classes, 'bcms_setimes')

### PREPARING BCMS TWITTER

In [ ]:
bcms_twitter_df2

In this setup, it is possible to have non-shibboleths annotated for linguistic features

(a word can be marked, just not specific to a single variety)

In [ ]:
f1 = bcms_twitter_df2[bcms_twitter_df2['shibboleth'] == 'yes'][['feature', 'label']]
f1

In [ ]:
f2 = bcms_twitter_df2[bcms_twitter_df2['shibboleth'] ==  'yes'][['feature2', 'label']].dropna()
f2['feature'] = f2['feature2']
f2 = f2.drop(columns=['feature2'])
f2

In [ ]:
feature_df = pd.concat([f1, f2])
feature_df['file'] = feature_df.index
feature_df = feature_df.reset_index().drop(columns=('index'))
feature_df = feature_df[feature_df.feature.str.contains('phon|lex|morph-d|morph-f|spel')]
feature_df = feature_df.join(bcms_setimes_methods_df, how='left', on='file').drop(columns=('file'))
feature_df

In [ ]:
methods = feature_df['method'].sort_values().unique()
methods

In [ ]:
classes = feature_df['label'].sort_values().unique()
classes

In [ ]:
bcms_twitter_levels_df = feature_df.groupby(['method', 'label', 'feature']).size().to_frame(name='count')
bcms_twitter_levels_df

In [ ]:
make_graphs_per_level(bcms_twitter_levels_df, methods, classes, 'bcms_twitter')

In [ ]:
make_sel_graph_per_level(bcms_twitter_levels_df, 'lime_0.1', classes, 'bcms_twitter')
#def make_sel_graph_per_level(df, method, classes, dataset):

## PREPARING FINNISH

In [ ]:
finnish_suomi24_df2

In [ ]:
feature_df = finnish_suomi24_df2[(finnish_suomi24_df2['lex_other_not'] == 'lex') | (finnish_suomi24_df2['lex_other_not'] == 'other')][['category', 'label', 'method']]
feature_df = feature_df[feature_df.category.str.contains('morph|lex|phon')]
feature_df

In [ ]:
methods = feature_df['method'].sort_values().unique()
methods

In [ ]:
classes = feature_df['label'].sort_values().unique()
classes

In [ ]:
finnish_suomi24_levels_df = feature_df.groupby(['method', 'label', 'category']).size().to_frame(name='count')
finnish_suomi24_levels_df

In [ ]:
make_graphs_per_level(finnish_suomi24_levels_df, methods, classes, 'finnish_suomi24-test')

In [ ]:
make_sel_graph_per_level(finnish_suomi24_levels_df, 'ig_0.6', classes, 'finnish_suomi24')

### PREPARING GREEK 

In [ ]:
greek_dialects_df2['feature'].sort_values().unique()

In [ ]:
feature_df = greek_dialects_df2[['feature', 'label', 'method']]
feature_df = feature_df[feature_df.feature.str.contains('Morph|Lex|Phon|Spell')]
feature_df

In [ ]:
methods = feature_df['method'].sort_values().unique()
methods

In [ ]:
classes = feature_df['label'].sort_values().unique()
classes

In [ ]:
greek_dialects_levels_df = feature_df.groupby(['method', 'label', 'feature']).size().to_frame(name='count')
greek_dialects_levels_df

In [ ]:
make_graphs_per_level(greek_dialects_levels_df, methods, classes, 'greek_dialects')

In [ ]:
make_sel_graph_per_level(greek_dialects_levels_df, 'ig_0.6', classes, 'greek_dialects')

### PREPARING JODEL

In [ ]:
german_jodel_df2[german_jodel_df2['shibboleth'] == 'yes']['code'].sort_values().unique()

In [ ]:
feature_df = german_jodel_df2[german_jodel_df2['shibboleth'] == 'yes'][['label', 'method', 'code']]
feature_df = feature_df[feature_df.code.str.contains('morph|lex|phon|spell')]
feature_df

In [ ]:
methods = feature_df['method'].sort_values().unique()
methods

In [ ]:
classes = feature_df['label'].sort_values().unique()
classes

In [ ]:
german_jodel_levels_df = feature_df.groupby(['method', 'label', 'code']).size().to_frame(name='count')
german_jodel_levels_df

In [ ]:
make_graphs_per_level(german_jodel_levels_df, methods, classes, 'german_jodel')

In [ ]:
make_sel_graph_per_level(german_jodel_levels_df, 'ig_0.1', classes, 'german_jodel')

### PREPARING SLIDE

In [ ]:
scandinavian_slide_df2

In [ ]:
feature_df = scandinavian_slide_df2[(scandinavian_slide_df2['LexShib'] == True) | \
                                    (scandinavian_slide_df2['NonLexShib'] == True)][['Morph', 'Phon_Spell', 'LexShib', 'method', 'label']]
feature_df['LexShib'] = feature_df['LexShib'].replace(True, 'Lex')
feature_df['Morph'] = feature_df['Morph'].replace(True, 'Morph')
feature_df['Phon_Spell'] = feature_df['Phon_Spell'].replace(True, 'Phon_Spell')
feature_df

In [ ]:
tmp1 = feature_df[['Morph', 'method', 'label']]
tmp1['feature'] = tmp1['Morph']
tmp1 = tmp1.drop(columns=['Morph'])
tmp1

In [ ]:
tmp2 = feature_df[['LexShib', 'method', 'label']]
tmp2['feature'] = tmp2['LexShib']
tmp2 = tmp2.drop(columns=['LexShib'])
tmp2

In [ ]:
tmp3 = feature_df[['Phon_Spell', 'method', 'label']]
tmp3['feature'] = tmp3['Phon_Spell']
tmp3 = tmp3.drop(columns=['Phon_Spell'])
tmp3

In [ ]:
feature_df = pd.concat([tmp1, tmp2, tmp3])
feature_df = feature_df[feature_df['feature'] != False]
feature_df

In [ ]:
methods = feature_df['method'].sort_values().unique()
methods

In [ ]:
classes = feature_df['label'].sort_values().unique()
classes

In [ ]:
scandinavian_slide_levels_df = feature_df.groupby(['method', 'label', 'feature']).size().to_frame(name='count')
scandinavian_slide_levels_df

In [ ]:
make_graphs_per_level(scandinavian_slide_levels_df, methods, classes, 'scandinavian_slide')